In [ ]:
import re
from unstructured.partition.pdf import partition_pdf
#表を読み込むサンプル
# ---------------------------------------------------------
# 1. 語の途中の改行だけ削除する（段落改行は維持）
# ---------------------------------------------------------
def fix_inline_linebreaks(text):
    lines = text.split("\n")
    fixed = []

    for line in lines:
        line = line.rstrip()

        # 前の行が句点で終わらない → 語の途中の改行と判断して結合
        if fixed and not fixed[-1].endswith("。"):
            fixed[-1] = fixed[-1] + line  # 改行削除して結合
        else:
            fixed.append(line)

    return "\n".join(fixed)


# ---------------------------------------------------------
# 2. ページの最後の行が「。」で終わるまで次ページを結合
# ---------------------------------------------------------
def merge_until_period(page_texts):
    merged_pages = []
    i = 0

    while i < len(page_texts):
        # 語の途中改行を修正（段落改行は維持）
        current_text = fix_inline_linebreaks(page_texts[i]["text"])
        lines = current_text.split("\n")

        last_line = lines[-1].rstrip()

        # 句点が付くまで次ページを結合
        while not last_line.endswith("。") and i + 1 < len(page_texts):
            next_text = fix_inline_linebreaks(page_texts[i + 1]["text"])
            next_lines = next_text.split("\n")

            if next_lines:
                # ★ 結合時は改行を削除して連結（あなたの要求）
                lines[-1] = last_line + next_lines[0]

                next_lines.pop(0)
                lines.extend(next_lines)

                last_line = lines[-1].rstrip()

            i += 1

        merged_pages.append("\n".join(lines))
        i += 1

    return merged_pages


# ---------------------------------------------------------
# 3. PDF → テキスト抽出 → PageBreak でページ分割
# ---------------------------------------------------------
elements = partition_pdf("sample.pdf")
full_text = "\n".join(el.text for el in elements)

pages_raw = re.split(r"<!--\s*PageBreak\s*-->", full_text)
page_texts = [{"page": idx, "text": page} for idx, page in enumerate(pages_raw, start=1)]

# ---------------------------------------------------------
# 4. 実行
# ---------------------------------------------------------
merged = merge_until_period(page_texts)

# ---------------------------------------------------------
# 5. 出力
# ---------------------------------------------------------
for page in merged:
    print("---- PAGE ----")
    print(page)


In [1]:
import re
from unstructured.partition.pdf import partition_pdf

# ---------------------------------------------------------
# 1. 不要ヘッダを削除する
# ---------------------------------------------------------
def remove_headers(text):
    # ロゴ（figure）
    text = re.sub(r"<figure>[\s\S]*?</figure>", "", text)

    # コメント版 PRESS RELEASE
    text = re.sub(r"<!--\s*PageHeader=\"PRESS RELEASE\"\s*-->", "", text)

    # テキスト版 PRESS RELEASE
    text = re.sub(r"PRESS RELEASE", "", text)

    # ★ 「各 位」〜「電話番号」までを削除（本文が続いていても削除）
    text = re.sub(
        r"各 位.*?電話番号",   # 「各 位」〜「電話番号」まで
        "",
        text,
        flags=re.DOTALL
    )

    # 日付（例：2021 年1月 12日）
    text = re.sub(r"\d{4}\s*年\s*\d+\s*月\s*\d+\s*日", "", text)

    return text


# ---------------------------------------------------------
# 2. 語の途中の改行だけ削除（段落改行は維持）
# ---------------------------------------------------------
def fix_inline_linebreaks(text):
    lines = text.split("\n")
    fixed = []

    for line in lines:
        line = line.rstrip()

        # 前の行が句点で終わらない → 語の途中の改行と判断して結合
        if fixed and not fixed[-1].endswith("。"):
            fixed[-1] = fixed[-1] + line  # 改行削除して結合
        else:
            fixed.append(line)

    return "\n".join(fixed)


# ---------------------------------------------------------
# 3. ページの最後が「。」で終わるまで次ページを結合
# ---------------------------------------------------------
def merge_until_period(page_texts):
    merged_pages = []
    i = 0

    while i < len(page_texts):
        current_text = fix_inline_linebreaks(page_texts[i]["text"])
        lines = current_text.split("\n")

        last_line = lines[-1].rstrip()

        while not last_line.endswith("。") and i + 1 < len(page_texts):
            next_text = fix_inline_linebreaks(page_texts[i + 1]["text"])
            next_lines = next_text.split("\n")

            if next_lines:
                # 結合時は改行削除
                lines[-1] = last_line + next_lines[0]

                next_lines.pop(0)
                lines.extend(next_lines)

                last_line = lines[-1].rstrip()

            i += 1

        merged_pages.append("\n".join(lines))
        i += 1

    return merged_pages


# ---------------------------------------------------------
# 4. PDF → テキスト抽出 → PageBreak でページ分割
# ---------------------------------------------------------
elements = partition_pdf("sample_ir_nxera.pdf")
full_text = "\n".join(el.text for el in elements)

# 不要ヘッダ削除
full_text = remove_headers(full_text)

# PageBreak でページ分割
pages_raw = re.split(r"<!--\s*PageBreak\s*-->", full_text)

page_texts = [{"page": idx, "text": page} for idx, page in enumerate(pages_raw, start=1)]

# ---------------------------------------------------------
# 5. 実行
# ---------------------------------------------------------
merged = merge_until_period(page_texts)

# ---------------------------------------------------------
# 6. 出力
# ---------------------------------------------------------
for page in merged:
    print("---- PAGE ----")
    print(page)


C:\Users\USER\miniforge3\envs\clense_test\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


---- PAGE ----
デュシェンヌ型筋ジストロフィー治療薬 vamorolone について 韓国において迅速な患者アクセスを支援する重要な規制上の指定を取得希少疾病用医薬品（ODD）の指定によりDMDが韓国において治療法が未確立で、 重大なアンメット・メディカル・ニーズを有する希少疾患と認定優先審査制度（GIFT）の指定により審査期間の短縮が可能vamoroloneの韓国での承認申請は2026年中を計画ネクセラファーマ株式会社（以下「当社」）は、本日、韓国食品医薬品安全処（MFDS）が、デュシェンヌ型筋ジストロフィー（DMD）の治療薬 vamorolone について、希少疾病用医薬品（Orphan Drug Designation, ODD）指定および優先審査制度（Global Innovative Products on Fast Track, GIFT）指定を付与したことをお知らせいたします。当社は 2026 年中に韓国における本剤の製造販売承認申請を行う計画です。
MFDS の希少疾病用医薬品（ODD）指定制度は、韓国国内の患者数が 2 万人以下の疾患を対象とする医薬品、または代替治療薬がない、もしくは既存治療と比較して安全性および有効性が大幅に改善された医薬品を対象とするものです。今回の ODD 指定は、vamorolone が MFDS の定める「希少疾病用医薬品指定に関する規定」の指定基準を満たしていると判断されたことを受けたものであり、MFDSが韓国において DMDを治療法が未確立の希少疾患として位置付けたことを意味します。
GIFT 指定は、生命を脅かす重篤な疾患または希少疾患を対象とする革新的医薬品の審査を迅速化する目的で2022 年 9 月に MFDS が導入した優先審査制度であり、通常 120 営業日とされる審査期間を最短 90 営業日まで短縮される可能性があり、韓国における革新的治療薬への患者さんのアクセスを加速する一助となることが期待されます。当社は、vamorolone が GIFT 指定を受けたことにより、韓国の DMD 患者さんへの本剤の早期アクセスが期待できるものと考えています。
当社子会社である Nxera Pharma Korea Co., Ltd.の代表取締役社長である MinBok Lee は次のように述べています。

In [2]:
from unstructured.partition.pdf import partition_pdf

elements = partition_pdf("sample_ir_nxera.pdf")
for el in elements:
    print(type(el), el.text[:80])


<class 'unstructured.documents.elements.Text'> 2026 年 6 月 2 日
<class 'unstructured.documents.elements.Title'> 各 位
<class 'unstructured.documents.elements.Title'> 本店所在地 東京都港区赤坂九丁目７番２号 会社名
<class 'unstructured.documents.elements.Text'> ネクセラファーマ株式会社 （コード番号 4565 東証プライム） 代表執行役社長 CEO クリストファー・カーギル IR 部 都築伸弥 03-5962-5718
<class 'unstructured.documents.elements.Title'> 代表者
<class 'unstructured.documents.elements.Title'> 問い合せ先 電話番号
<class 'unstructured.documents.elements.NarrativeText'> デュシェンヌ型筋ジストロフィー治療薬 vamorolone について 韓国において迅速な患者アクセスを支援する重要な規制上の指定を取得
<class 'unstructured.documents.elements.Title'> 希少疾病用医薬品（ODD）の指定によりDMDが韓国において治療法が未確立で、 重大なアンメット・メディカル・ニーズを有する希少疾患と認定
<class 'unstructured.documents.elements.Title'> 優先審査制度（GIFT）の指定により審査期間の短縮が可能
<class 'unstructured.documents.elements.Title'> vamoroloneの韓国での承認申請は2026年中を計画
<class 'unstructured.documents.elements.Title'> ネクセラファーマ株式会社（以下「当社」）は、本日、韓国食品医薬品安全処（MFDS）が、デュシェンヌ型筋
<class 'unstructured.documents.elements.Title'> ジストロフィー（DMD）の治療薬 vamorolone につ

In [ ]:
import shutil
print(shutil.which("pdfinfo"))

In [5]:
import subprocess

try:
    subprocess.run(["pdfinfo", "-h"], check=True)
    print("pdfinfo OK")
except Exception as e:
    print("pdfinfo NG", e)


pdfinfo OK


In [1]:
import shutil
print(shutil.which("tesseract"))


C:\Program Files\Tesseract-OCR\tesseract.EXE
